In [ ]:
"""
SBI based posterior inference for alpha and rho parameters of the gamma distribution of rate variation across sites.
This script uses existing training data (features + true parameters) to train a neural network model that can predict alpha and rho from features.
1. Trains a neural network using sbi (NPE) to learn the posterior distribution of alpha and rho given the features.
2. present the results with the script visualize_posteriors.py, which will show how well the model can recover the true parameters on a test set and visualize the learned posterior distributions.
"""
from dataclasses import dataclass
import math
import random
import torch
from sbi.utils import BoxUniform
from sbi.inference import NPE
from sbi.inference import simulate_for_sbi
from sbi.analysis import pairplot


from ete3 import Tree
import numpy as np

from msasim import protocol, simulator as sim
from msasim.msa import Msa
from msasim.distributions import ZipfDistribution
from msasim.constants import SITE_RATE_MODELS
from msasim.constants import MODEL_CODES

import prior
from features_calculator import calculate_all_features

@dataclass
class SimulationParams:
    """Parameters controlling the shape of simulated MSAs."""
    min_taxa: int = 20
    max_taxa: int = 200
    min_seq_length: int = 500
    max_seq_length: int = 1000

params = SimulationParams()  # Use default simulation parameters for taxa and sequence length ranges



def generate_random_tree(n_taxa: int, scale: float) -> Tree:
    """Generate a random ultrametric tree with n_taxa leaves."""

    tree = Tree()
    tree.populate(n_taxa, random_branches=True)
    for node in tree.traverse():
        if node.dist == 0:
            continue
        node.dist = np.random.exponential(scale=scale)
    return tree


def setup_sim(tree: Tree) -> sim.Simulator:
    """
    Setup the simulator for a given tree and seed.

    Args:
        tree: ete3.Tree object
        sim_seed: Seed for this simulation

    Returns:
        msasim.Simulator object
    """
    newick_string = tree.write(format=1)
    simulation_protocol = protocol.SimProtocol(newick_string)
    # indel model parameters based on mammalian rates from https://doi.org/10.1093/bioinformatics/btaf686
    simulation_protocol.set_insertion_rates(0.007)
    simulation_protocol.set_deletion_rates(0.035)
    simulation_protocol.set_insertion_length_distributions(ZipfDistribution(p=1.53, truncation=50))
    simulation_protocol.set_deletion_length_distributions(ZipfDistribution(p=1.11, truncation=50))
    simulation_protocol.set_site_rate_model(SITE_RATE_MODELS.INDEL_AWARE)
    simulator = sim.Simulator(simulation_protocol, simulation_type=sim.SIMULATION_TYPE.PROTEIN)
    return simulator


prior_dist = BoxUniform(
    low=torch.tensor([prior.ALPHA_RANGE[0], prior.RHO_RANGE[0], math.log10(prior.TREE_SCALE_RANGE[0])]),
    high=torch.tensor([prior.ALPHA_RANGE[1], prior.RHO_RANGE[1], math.log10(prior.TREE_SCALE_RANGE[1])])
)    



def simulate(theta: torch.Tensor):
    """

    Simulate a single an MSA based on a random tree.

    Args:
        theta: torch.Tensor of shape (3,) containing the parameters (alpha, rho, tree_scale) for this simulation
    Returns:
        tuple: 
            - features: torch.Tensor of shape (num_features,) containing the calculated features for this simulation
    """

    # go through all params in tensor and simulate for each then return all features as tensor
    # loop through each row of theta and simulate an MSA for each set of parameters, then save features for each MSA and return as tensor
    features_list = []
    for i in range(theta.shape[0]):
        true_alpha, true_rho, true_tree_scale = theta[i].tolist()
        # convert log10 tree_scale back to linear scale
        true_tree_scale = 10 ** true_tree_scale


        n_taxa = random.randint(params.min_taxa, params.max_taxa)
        simulator = setup_sim(generate_random_tree(n_taxa=n_taxa, scale=true_tree_scale))  # Example simulator for testing

        seq_length = random.randint(params.min_seq_length, params.max_seq_length)

        simulator.protocol.set_sequence_size(seq_length)
        simulator.set_replacement_model(
            model=MODEL_CODES.WAG,
            gamma_parameters_alpha=true_alpha,
            gamma_parameters_categories=8,
            site_rate_correlation=true_rho
        )

        msa: Msa = simulator()
        sequences = [msa.get_msa_row(i).split("\n")[1] for i in range(msa.get_num_sequences())]

        stats = calculate_all_features(sequences)
        features_list.append(torch.tensor(list(stats.values()), dtype=torch.float32))

    # return tensor of features for this simulation
    return torch.stack(features_list)

In [ ]:
theta_0 = prior_dist.sample((1,))
feature_0 = simulate(theta_0)

# print theta and features for this example simulation
print("Sampled parameters (alpha, rho, tree_scale):", theta_0)


In [ ]:

# print("Calculated features:", feature_0)
num_simulations = 15000
thetas, features = simulate_for_sbi(simulate, prior_dist, 
                                    num_simulations=num_simulations,
                                    num_workers=7)  # Simulate MSAs and calculate features for each

inference = NPE(prior=prior_dist)


inference.append_simulations(thetas, features).train()

posterior = inference.build_posterior()
print(posterior)

x_obs = feature_0
samples = posterior.sample((1000,), x=x_obs)

# mean and std of samples
print("Posterior samples mean:", samples.mean(dim=0))
print("Posterior samples std:", samples.std(dim=0))

In [ ]:

_ = pairplot(samples.numpy(),
              limits=[[0, 2], [0, 1], [np.log10(0.001), np.log10(0.2)]],
              labels=["alpha", "rho", "tree_scale"], show_titles=True)

In [ ]:
# test accuracy of model on a new simulation
true_parameters = prior_dist.sample((200,))
infered_params = []

for i in true_parameters:
    print("Testing on parameters (alpha, rho, tree_scale):", i)
    features_test = simulate(i.unsqueeze(0))
    posterior_samples_test = posterior.sample((1000,), x=features_test, show_progress_bars=False)
    infered_params.append(posterior_samples_test.median(dim=0).values)

infered_params = torch.stack(infered_params)


In [ ]:
# compute R^2 between true parameters (theta_test) and posterior samples median
# fore each parameter (alpha, rho, tree_scale)

R2_alpha = 1 - torch.sum((infered_params[:, 0] - true_parameters[:, 0]) ** 2) / torch.sum((true_parameters[:, 0] - true_parameters[:, 0].median()) ** 2)
R2_rho = 1 - torch.sum((infered_params[:, 1] - true_parameters[:, 1]) ** 2) / torch.sum((true_parameters[:, 1] - true_parameters[:, 1].median()) ** 2)
R2_tree_scale = 1 - torch.sum((infered_params[:, 2] - true_parameters[:, 2]) ** 2) / torch.sum((true_parameters[:, 2] - true_parameters[:, 2].median()) ** 2)

print("Test set R^2 (alpha):", R2_alpha)
print("Test set R^2 (rho):", R2_rho)
print("Test set R^2 (tree_scale):", R2_tree_scale)


In [ ]:
# scatter plot of true vs predicted parameters for test set
import matplotlib.pyplot as plt
plt.figure(figsize=(15, 5))
plt.subplot(1, 3, 1)
plt.scatter(true_parameters[:, 0].numpy(), infered_params[:, 0].numpy(), alpha=0.5)
plt.xlabel("True alpha")
plt.ylabel("Predicted alpha")
plt.subplot(1, 3, 2)
plt.scatter(true_parameters[:, 1].numpy(), infered_params[:, 1].numpy(), alpha=0.5)
plt.xlabel("True rho")
plt.ylabel("Predicted rho")
plt.subplot(1, 3, 3)
plt.scatter(true_parameters[:, 2].numpy(), infered_params[:, 2].numpy(), alpha=0.5)
plt.xlabel("True log10(tree_scale)")
plt.ylabel("Predicted log10(tree_scale)")
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

FEATURE_NAMES = [
    'avg_entropy', 'entropy_variance', 'max_entropy', 'lag1_autocorr',
    'entropy_skewness', 'entropy_kurtosis', 'bimodality_coefficient', 'gamma_shape_entropy',
    'entropy_bin_0', 'entropy_bin_1', 'entropy_bin_2', 'entropy_bin_3', 'entropy_bin_4',
    'entropy_bin_5', 'entropy_bin_6', 'entropy_bin_7', 'entropy_bin_8',
    'lag2_autocorr', 'lag3_autocorr', 'lag4_autocorr', 'lag5_autocorr', 'lag6_autocorr',
    'lag7_autocorr', 'lag8_autocorr', 'lag9_autocorr', 'lag10_autocorr',
    'high_run_mean', 'high_run_var', 'high_run_max', 'high_run_count',
    'low_run_mean',  'low_run_var',  'low_run_max',  'low_run_count',
    'avg_gap_size', 'msa_len', 'msa_max_len', 'msa_min_len', 'tot_num_gaps',
    'num_gaps_len_one', 'num_gaps_len_two', 'num_gaps_len_three', 'num_gaps_len_at_least_four',
    'n_taxa'
]
PARAM_NAMES = ['alpha', 'rho', 'log10_tree_scale']

F = features.numpy()
T = thetas.numpy()
# add new features and parameters from new_thetas and new_features

corr = np.array([[np.corrcoef(F[:, i], T[:, j])[0, 1]
                  for j in range(3)]
                 for i in range(len(FEATURE_NAMES))])
print(corr)
fig, ax = plt.subplots(figsize=(5, 14))
im = ax.imshow(corr, aspect='auto', cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(3)); ax.set_xticklabels(PARAM_NAMES)
ax.set_yticks(range(len(FEATURE_NAMES))); ax.set_yticklabels(FEATURE_NAMES, fontsize=8)
plt.colorbar(im, ax=ax, fraction=0.03)
plt.title('Feature–parameter correlations')
plt.tight_layout()
plt.show()

In [ ]:
#infer posterior for empirical MSA
x_empirical = torch.tensor(list(empirical_features.values()), dtype=torch.float32)
empirical_posterior_samples = posterior.sample((1000,), x=x_empirical)
print("Empirical posterior samples:", empirical_posterior_samples.median(0))

_ = pairplot(empirical_posterior_samples.numpy(),
              limits=[[0, 2], [0, 1], [np.log10(0.001), np.log10(0.2)]],
              labels=["alpha", "rho", "tree_scale"], show_titles=True)

In [ ]:
num_rounds = 2
num_simulations = 1000

inference_seq = NPE(prior=prior_dist)
proposal = posterior.set_default_x(x_obs)

for i in range(num_rounds):
    theta, x = simulate_for_sbi(simulate, proposal, num_simulations=num_simulations, num_workers=7)
    density_estimator = inference_seq.append_simulations(theta, x, proposal=proposal).train()
    posterior_refined = inference_seq.build_posterior(density_estimator)
    proposal = posterior_refined.set_default_x(x_obs)

samples_refined = posterior_refined.sample((1000,), x=x_obs)

_ = pairplot(samples_refined.numpy(),
             limits=[[0, 2], [0, 1], [np.log10(0.001), np.log10(0.2)]],
             labels=["alpha", "rho", "tree_scale"], show_titles=True)